In [8]:
# load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
# create an API client using the API key from the environment variable
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-5-5"  # Define the model to use for the API calls. You can change this to any model you want to use.

In [12]:
# --------------- Helper Functions -----------------------------------------------
#From Multi-Turn conversation example in the "Build with the Anthropic API" course
# Add a user message to the messages list
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# Add an assistant message to the messages list
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Define a function to send messages to the API and get a response, set system prompt if provided, None if not provided
def chat(messages, system = None):
    # Set up the parameters for the API request
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        # Adding temperature parameter to control randomness of the model's output  
        #"temperature": temperature

        # Disabling thinking for Haiku 5-5, which is thinking=adaptive by default.
        # This Prevents the thinking.block error
        "thinking": {"type": "disabled"} 
    }
    
    # if a system prompt is provided, add it to the parameters   
    if system:
        # Add the system prompt to the parameters
        params["system"] = system
    
    # Make the API request and return the assistant's response, parameters are unpacked using the ** operator
    message = client.messages.create(**params)
    #return message.content[0].text
    
    # From Gemini:
    # Safe parsing pattern: filters out non-text blocks to prevent future crashes
    return "".join(
      [block.text for block in message.content if block.type == "text"]
    )

### These helper functions enable you to build a multi-turn conversation with the API. You can add user and assistant messages to the messages list, and then call the chat function to get a response from the API based on the conversation history.
# --------------------------------------------------------------

In [13]:
messages = []

#This one is for the system prompt test
#add_user_message(messages, "How do I solve the equation 2x + 3 = 7?")

#This one is for the temperature test
add_user_message(
    messages, 
    "What is the solution ox 2x + 3 = 25",
)

#answer = chat(messages)
answer = chat(messages, system = "You are a patient math tutor. Do not directly give the answer to the user, but instead guide them through the problem-solving process with hints and questions.")

#anwswer for system prompt test
#print("Assistant:", anwswer)
print("Assistant:", answer)

#Answer for temperature test
#answer

Assistant: Let's work through this together. You have the equation 2x + 3 = 25, and our goal is to find the value of x that makes it true.

Here's a question to start: right now, the 3 is being added to 2x. What operation could we use on both sides of the equation to "undo" that addition?

Give it a try, and tell me what you get for the new equation. Then we'll take the next step.


In [5]:
#System prompt to set the behavior of the assistant
system_prompt = """You are a patient math tutor. Do not directly give the answer to the user,
but instead guide them through the problem-solving process with hints and questions."""

client.messages.create(
    model = "claude-haiku-4-5",
    messages = messages,
    max_tokens = 1000,
    system=system_prompt,
    
)

PermissionDeniedError: Error code: 403 - {'error': {'message': "The requested model 'claude-haiku-4-5' is not available for this API key, or the model name is invalid. Check the models available to you and try again.", 'type': 'team_model_access_denied', 'param': 'model', 'code': '403'}}